# Week 12 Surrogate Alternatives Analysis

**Purpose:** With 2 queries remaining per function, evaluate whether alternative surrogates
could improve suggestions for stale functions. Four analyses:

1. **NN-64×2 surrogate for F6** — NN LOO R² = 0.734 vs GP 0.690; can the NN break a 3-week plateau?
2. **Local GP (TuRBO-style) for F3** — fit GP on k-nearest to best; avoids pathological global length-scales
3. **Model-free weighted centroid** — zero-model baseline for all 8 functions
4. **RF feature importance update for F8** — validate dimension mask with n=51

**Data:** Initial .npy observations + W1–W11 portal submissions (W11 results now returned).

---

In [1]:
import numpy as np
import json
import warnings
from pathlib import Path

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    Matern, ConstantKernel as C, RBF, RationalQuadratic,
)
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")
np.random.seed(42)

ROOT = Path("..") 
with open(ROOT / "capstone_history.json") as f:
    hist = json.load(f)


def load_function(fn_id):
    """Return (X, Y, dims) combining initial data and non-zero submissions."""
    X_init = np.load(ROOT / f"initial_data/function_{fn_id}/initial_inputs.npy")
    Y_init = np.load(ROOT / f"initial_data/function_{fn_id}/initial_outputs.npy").ravel()
    d = hist[str(fn_id)]
    valid = [(i, d["Y"][i]) for i in range(len(d["Y"])) if d["Y"][i] != 0]
    if valid:
        X_sub = np.array([d["X"][i] for i, _ in valid])
        Y_sub = np.array([y for _, y in valid])
        X = np.vstack([X_init, X_sub])
        Y = np.hstack([Y_init, Y_sub])
    else:
        X, Y = X_init, Y_init
    return X, Y, X.shape[1]


def standardize(Y):
    return (Y - Y.mean()) / (Y.std() + 1e-12)


def loo_r2(model_fn, X, Y, scale_x=False):
    """Manual leave-one-out R²."""
    n = len(Y)
    preds = np.zeros(n)
    for i in range(n):
        mask = np.ones(n, dtype=bool)
        mask[i] = False
        X_tr, Y_tr = X[mask], Y[mask]
        X_te = X[i : i + 1]
        if scale_x:
            sc = StandardScaler()
            X_tr = sc.fit_transform(X_tr)
            X_te = sc.transform(X_te)
        m = model_fn()
        m.fit(X_tr, Y_tr)
        preds[i] = m.predict(X_te)[0]
    ss_res = np.sum((Y - preds) ** 2)
    ss_tot = np.sum((Y - Y.mean()) ** 2)
    return 1 - ss_res / ss_tot if ss_tot > 0 else 0.0


print("Data loaded. Functions available: 1–8")
for fn_id in range(1, 9):
    X, Y, dims = load_function(fn_id)
    best_y = Y.max()
    print(f"  F{fn_id}: n={len(Y)}, dims={dims}, best Y={best_y:.6g}")

Data loaded. Functions available: 1–8
  F1: n=21, dims=2, best Y=1.64327e-07
  F2: n=21, dims=2, best Y=0.725983
  F3: n=26, dims=3, best Y=-0.00831266
  F4: n=41, dims=4, best Y=0.367409
  F5: n=31, dims=4, best Y=4125.88
  F6: n=31, dims=5, best Y=-0.246173
  F7: n=41, dims=6, best Y=2.58832
  F8: n=51, dims=8, best Y=9.87527


---
## 1. NN-64×2 Surrogate for F6

F6 (5D Cake Recipe) has been stale for 3 weeks (best at W8). The preliminary scan showed
NN-64×2 reaching LOO R² = 0.734 vs GP's 0.690. Here we:

1. Confirm the LOO R² gap with multiple random seeds
2. Train a final NN on all data and generate a suggestion via posterior mean maximisation
3. Compare the NN suggestion to the GP suggestion and model-free centroid

In [2]:
X6, Y6, dims6 = load_function(6)
Y6s = standardize(Y6)
n6 = len(Y6)

print(f"F6: n={n6}, dims={dims6}")
print(f"Best Y = {Y6.max():.6f} at X = {X6[np.argmax(Y6)]}")
print()

# --- GP baseline (Matérn 5/2) ---
gp_fn = lambda: GaussianProcessRegressor(
    kernel=C(1.0) * Matern(length_scale=1.0, nu=2.5),
    alpha=1e-6, normalize_y=False, n_restarts_optimizer=5,
)
r2_gp = loo_r2(gp_fn, X6, Y6s)
print(f"GP Matérn 5/2  LOO R² = {r2_gp:.4f}")

# --- NN-64×2 across multiple seeds ---
nn_r2s = []
for seed in range(5):
    nn_fn = lambda s=seed: MLPRegressor(
        hidden_layer_sizes=(64, 32),
        max_iter=2000,
        random_state=s,
        alpha=0.01,
        early_stopping=False,
    )
    r2 = loo_r2(nn_fn, X6, Y6s, scale_x=True)
    nn_r2s.append(r2)
    print(f"NN-64×2 seed={seed} LOO R² = {r2:.4f}")

print(f"\nNN-64×2 mean LOO R² = {np.mean(nn_r2s):.4f} ± {np.std(nn_r2s):.4f}")
print(f"GP Matérn 5/2  LOO R² = {r2_gp:.4f}")
print(f"Delta = {np.mean(nn_r2s) - r2_gp:+.4f}")

F6: n=31, dims=5
Best Y = -0.246173 at X = [0.472233 0.407216 0.735084 0.782124 0.017818]



GP Matérn 5/2  LOO R² = 0.6899


NN-64×2 seed=0 LOO R² = 0.6651


NN-64×2 seed=1 LOO R² = 0.7087


NN-64×2 seed=2 LOO R² = 0.6886


NN-64×2 seed=3 LOO R² = 0.6827


NN-64×2 seed=4 LOO R² = 0.7185

NN-64×2 mean LOO R² = 0.6927 ± 0.0190
GP Matérn 5/2  LOO R² = 0.6899
Delta = +0.0028


In [3]:
# --- Generate NN suggestion via argmax of mean prediction ---
scaler6 = StandardScaler()
X6_scaled = scaler6.fit_transform(X6)

# Train ensemble of 10 NNs for robustness
nn_ensemble = []
for seed in range(10):
    nn = MLPRegressor(
        hidden_layer_sizes=(64, 32),
        max_iter=2000,
        random_state=seed,
        alpha=0.01,
        early_stopping=False,
    )
    nn.fit(X6_scaled, Y6s)
    nn_ensemble.append(nn)

# Generate candidates within F6's confirmed basin
bounds_f6 = [(0.35, 0.55), (0.30, 0.55), (0.65, 0.85), (0.70, 0.90), (0.0, 0.05)]
lows = np.array([b[0] for b in bounds_f6])
highs = np.array([b[1] for b in bounds_f6])
candidates = np.random.uniform(lows, highs, (10000, dims6))

# NN ensemble mean prediction
cand_scaled = scaler6.transform(candidates)
nn_preds = np.array([nn.predict(cand_scaled) for nn in nn_ensemble])
nn_mean = nn_preds.mean(axis=0)
nn_std = nn_preds.std(axis=0)

nn_best_idx = np.argmax(nn_mean)
nn_suggestion = candidates[nn_best_idx]
print(f"NN ensemble suggestion: {[round(v, 4) for v in nn_suggestion]}")
print(f"  Predicted Y (std units): {nn_mean[nn_best_idx]:.4f} ± {nn_std[nn_best_idx]:.4f}")

# GP suggestion for comparison
gp6 = GaussianProcessRegressor(
    kernel=C(1.0) * Matern(length_scale=1.0, nu=2.5),
    alpha=1e-6, normalize_y=False, n_restarts_optimizer=5,
)
gp6.fit(X6, Y6s)
gp_mean, gp_std = gp6.predict(candidates, return_std=True)
gp_best_idx = np.argmax(gp_mean)
gp_suggestion = candidates[gp_best_idx]
print(f"\nGP suggestion:          {[round(v, 4) for v in gp_suggestion]}")
print(f"  Predicted Y (std units): {gp_mean[gp_best_idx]:.4f} ± {gp_std[gp_best_idx]:.4f}")

# Compare to best known
best_x6 = X6[np.argmax(Y6)]
print(f"\nBest known:             {[round(v, 4) for v in best_x6]}")
print(f"  NN dist from best:  {np.linalg.norm(nn_suggestion - best_x6):.4f}")
print(f"  GP dist from best:  {np.linalg.norm(gp_suggestion - best_x6):.4f}")

NN ensemble suggestion: [np.float64(0.3528), np.float64(0.3017), np.float64(0.698), np.float64(0.7202), np.float64(0.013)]
  Predicted Y (std units): 1.5515 ± 0.0655

GP suggestion:          [np.float64(0.4643), np.float64(0.4398), np.float64(0.7114), np.float64(0.7195), np.float64(0.0448)]
  Predicted Y (std units): 1.6697 ± 0.0826

Best known:             [np.float64(0.4722), np.float64(0.4072), np.float64(0.7351), np.float64(0.7821), np.float64(0.0178)]
  NN dist from best:  0.1750
  GP dist from best:  0.0796


---
## 2. Local GP (TuRBO-style) for F3

F3's global GP has LOO R² < 0.10 with pathological length-scales (D1 = 4.18).
TuRBO's core insight: fit the GP only on the **k nearest observations** to the current
best. This:

- Avoids long-range extrapolation that inflates length-scales
- Focuses the GP on local curvature near the optimum
- Is equivalent to a trust-region approach without the full BoTorch machinery

We compare global vs local GP length-scales, LOO R², and generated suggestions.

In [4]:
X3, Y3, dims3 = load_function(3)
Y3s = standardize(Y3)
n3 = len(Y3)
best_idx3 = np.argmax(Y3)
best_x3 = X3[best_idx3]

print(f"F3: n={n3}, dims={dims3}")
print(f"Best Y = {Y3.max():.6f} at X = {[round(v, 4) for v in best_x3]}")
print()

# --- Global GP ---
gp_global = GaussianProcessRegressor(
    kernel=C(1.0) * Matern(length_scale=np.ones(dims3), nu=2.5),
    alpha=1e-6, normalize_y=False, n_restarts_optimizer=5,
)
gp_global.fit(X3, Y3s)
global_ls = np.exp(gp_global.kernel_.theta)
print(f"Global GP length-scales: {[round(v, 3) for v in global_ls]}")

# Global GP argmax
cands3 = np.random.uniform(0, 1, (10000, dims3))
mu_global = gp_global.predict(cands3)
global_argmax = cands3[np.argmax(mu_global)]
print(f"Global GP argmax: {[round(v, 3) for v in global_argmax]}")
print(f"  Distance from best: {np.linalg.norm(global_argmax - best_x3):.4f}")
print()

# --- Local GP (k-nearest) for several k values ---
dists_from_best = np.linalg.norm(X3 - best_x3, axis=1)

for k in [10, 15, 20]:
    nearest = np.argsort(dists_from_best)[:k]
    X_local = X3[nearest]
    Y_local = Y3s[nearest]

    gp_local = GaussianProcessRegressor(
        kernel=C(1.0) * Matern(length_scale=np.ones(dims3), nu=2.5),
        alpha=1e-6, normalize_y=False, n_restarts_optimizer=5,
    )
    gp_local.fit(X_local, Y_local)
    local_ls = np.exp(gp_local.kernel_.theta)

    # Local GP argmax (within search bounds)
    bounds_f3 = [(0.35, 0.60), (0.36, 0.56), (0.43, 0.55)]
    lows3 = np.array([b[0] for b in bounds_f3])
    highs3 = np.array([b[1] for b in bounds_f3])
    cands_local = np.random.uniform(lows3, highs3, (10000, dims3))

    mu_local = gp_local.predict(cands_local)
    local_argmax = cands_local[np.argmax(mu_local)]

    print(f"Local GP (k={k}):")
    print(f"  Length-scales: {[round(v, 3) for v in local_ls]}")
    print(f"  Argmax: {[round(v, 3) for v in local_argmax]}")
    print(f"  Distance from best: {np.linalg.norm(local_argmax - best_x3):.4f}")
    print()

F3: n=26, dims=3
Best Y = -0.008313 at X = [np.float64(0.5113), np.float64(0.4347), np.float64(0.4839)]



Global GP length-scales: [np.float64(1.054), np.float64(100000.0), np.float64(0.002), np.float64(4643.46)]
Global GP argmax: [np.float64(0.844), np.float64(0.434), np.float64(0.99)]
  Distance from best: 0.6062



Local GP (k=10):
  Length-scales: [np.float64(0.359), np.float64(0.433), np.float64(52579.948), np.float64(0.036)]
  Argmax: [np.float64(0.6), np.float64(0.497), np.float64(0.492)]
  Distance from best: 0.1085



Local GP (k=15):
  Length-scales: [np.float64(0.242), np.float64(1.566), np.float64(0.46), np.float64(0.061)]
  Argmax: [np.float64(0.594), np.float64(0.478), np.float64(0.495)]
  Distance from best: 0.0944



Local GP (k=20):
  Length-scales: [np.float64(0.264), np.float64(34036.464), np.float64(1.211), np.float64(0.015)]
  Argmax: [np.float64(0.378), np.float64(0.56), np.float64(0.477)]
  Distance from best: 0.1830



In [5]:
# --- LOO R² comparison: global vs local ---
print("=== LOO R² comparison for F3 ===\n")

# Global
gp_fn3 = lambda: GaussianProcessRegressor(
    kernel=C(1.0) * Matern(length_scale=1.0, nu=2.5),
    alpha=1e-6, normalize_y=False, n_restarts_optimizer=5,
)
r2_global = loo_r2(gp_fn3, X3, Y3s)
print(f"Global GP (all n={n3}):  LOO R² = {r2_global:.4f}")

# Local (k=15) — LOO within the local subset
k = 15
nearest = np.argsort(dists_from_best)[:k]
X_local = X3[nearest]
Y_local = Y3s[nearest]
r2_local = loo_r2(gp_fn3, X_local, Y_local)
print(f"Local GP (k={k}):       LOO R² = {r2_local:.4f}")

# NN for comparison
nn_fn3 = lambda: MLPRegressor(
    hidden_layer_sizes=(64, 32), max_iter=2000, random_state=42, alpha=0.01,
)
r2_nn3 = loo_r2(nn_fn3, X3, Y3s, scale_x=True)
print(f"NN-64×2 (all n={n3}):   LOO R² = {r2_nn3:.4f}")

# RF
rf_fn3 = lambda: RandomForestRegressor(n_estimators=100, max_depth=4, random_state=42)
r2_rf3 = loo_r2(rf_fn3, X3, Y3s)
print(f"RF (all n={n3}):        LOO R² = {r2_rf3:.4f}")

=== LOO R² comparison for F3 ===



Global GP (all n=26):  LOO R² = -0.0142


Local GP (k=15):       LOO R² = 0.5114


NN-64×2 (all n=26):   LOO R² = 0.2742


RF (all n=26):        LOO R² = 0.2439


---
## 3. Model-Free Weighted Centroid — All Functions

The simplest possible "surrogate": take the top-k observations weighted by Y and
compute their centroid. This requires zero model assumptions and serves as a lower
bound that any surrogate should beat.

For each function we compute the weighted centroid of the top-5 observations and
measure its distance from the best-known point.

In [6]:
print("=== Model-free weighted centroid (top-5) for all functions ===\n")
print(f"{'Fn':<4} {'n':>3} {'d':>2}  {'Centroid':>50}  {'Dist→Best':>10}  {'Best X':>50}")
print("-" * 130)

centroid_results = {}

for fn_id in range(1, 9):
    X, Y, dims = load_function(fn_id)
    best_idx = np.argmax(Y)
    best_x = X[best_idx]

    top_k = 5
    top_idx = np.argsort(Y)[-top_k:]
    top_x = X[top_idx]
    top_y = Y[top_idx]

    weights = top_y - top_y.min() + 1e-8
    weights /= weights.sum()
    centroid = np.average(top_x, weights=weights, axis=0)

    dist = np.linalg.norm(centroid - best_x)
    centroid_results[fn_id] = {"centroid": centroid, "dist": dist, "best_x": best_x}

    c_str = str([round(v, 4) for v in centroid])
    b_str = str([round(v, 4) for v in best_x])
    print(f"F{fn_id:<3} {len(Y):>3} {dims:>2}  {c_str:>50}  {dist:>10.4f}  {b_str:>50}")

print()
print("Interpretation: smaller distance → the top-5 cluster tightly around the best.")
print("Large distance → outlier best or multi-modal top-5.")

=== Model-free weighted centroid (top-5) for all functions ===

Fn     n  d                                            Centroid   Dist→Best                                              Best X
----------------------------------------------------------------------------------------------------------------------------------
F1    21  2            [np.float64(0.6935), np.float64(0.7164)]      0.0098              [np.float64(0.691), np.float64(0.707)]
F2    21  2            [np.float64(0.6993), np.float64(0.9345)]      0.0021            [np.float64(0.6987), np.float64(0.9324)]
F3    26  3  [np.float64(0.5054), np.float64(0.4406), np.float64(0.4946)]      0.0136  [np.float64(0.5113), np.float64(0.4347), np.float64(0.4839)]
F4    41  4  [np.float64(0.4375), np.float64(0.4309), np.float64(0.3623), np.float64(0.3815)]      0.0075  [np.float64(0.4384), np.float64(0.4311), np.float64(0.355), np.float64(0.3801)]
F5    31  4  [np.float64(0.3259), np.float64(0.9586), np.float64(0.9849), np.float64(0

In [7]:
# Detailed per-function centroid analysis for stale functions
print("=== Centroid vs GP vs NN suggestions for stale functions ===\n")

for fn_id in [2, 4, 6]:
    X, Y, dims = load_function(fn_id)
    Y_s = standardize(Y)
    best_x = X[np.argmax(Y)]
    centroid = centroid_results[fn_id]["centroid"]

    # GP suggestion (mean only, within basin)
    gp = GaussianProcessRegressor(
        kernel=C(1.0) * Matern(length_scale=1.0, nu=2.5),
        alpha=1e-6, normalize_y=False, n_restarts_optimizer=5,
    )
    gp.fit(X, Y_s)

    cands = np.random.uniform(0, 1, (10000, dims))
    mu_gp = gp.predict(cands)
    gp_sug = cands[np.argmax(mu_gp)]

    print(f"F{fn_id} (best = {Y.max():.6g}):")
    print(f"  Best known:   {[round(v, 4) for v in best_x]}")
    print(f"  Centroid:     {[round(v, 4) for v in centroid]}  dist={np.linalg.norm(centroid - best_x):.4f}")
    print(f"  GP argmax:    {[round(v, 4) for v in gp_sug]}  dist={np.linalg.norm(gp_sug - best_x):.4f}")
    print()

=== Centroid vs GP vs NN suggestions for stale functions ===

F2 (best = 0.725983):
  Best known:   [np.float64(0.6987), np.float64(0.9324)]
  Centroid:     [np.float64(0.6993), np.float64(0.9345)]  dist=0.0021
  GP argmax:    [np.float64(0.6986), np.float64(0.9382)]  dist=0.0057



F4 (best = 0.367409):
  Best known:   [np.float64(0.4384), np.float64(0.4311), np.float64(0.355), np.float64(0.3801)]
  Centroid:     [np.float64(0.4375), np.float64(0.4309), np.float64(0.3623), np.float64(0.3815)]  dist=0.0075
  GP argmax:    [np.float64(0.4063), np.float64(0.4389), np.float64(0.3663), np.float64(0.3969)]  dist=0.0387



F6 (best = -0.246173):
  Best known:   [np.float64(0.4722), np.float64(0.4072), np.float64(0.7351), np.float64(0.7821), np.float64(0.0178)]
  Centroid:     [np.float64(0.4507), np.float64(0.4161), np.float64(0.7223), np.float64(0.776), np.float64(0.0289)]  dist=0.0294
  GP argmax:    [np.float64(0.449), np.float64(0.4672), np.float64(0.7834), np.float64(0.688), np.float64(0.0616)]  dist=0.1313



---
## 4. RF Feature Importance Update for F8

F8's dimension mask (dropping D6, D8) was based on analysis at n ≈ 44. With n = 51,
re-run RF feature importance to validate or update the mask. Also compute Spearman
correlations between each dimension and Y among the top-10 observations.

In [8]:
X8, Y8, dims8 = load_function(8)
Y8s = standardize(Y8)
n8 = len(Y8)

print(f"F8: n={n8}, dims={dims8}")
print(f"Best Y = {Y8.max():.6f}\n")

# --- RF feature importance ---
rf8 = RandomForestRegressor(n_estimators=500, max_depth=5, random_state=42)
rf8.fit(X8, Y8s)

importances = rf8.feature_importances_
print("RF feature importance (n=500 trees, max_depth=5):")
for i in range(dims8):
    bar = "█" * int(importances[i] * 50)
    mask_status = "(MASKED)" if i in [5, 7] else ""
    print(f"  D{i+1}: {importances[i]:.4f} {bar} {mask_status}")

print(f"\nCurrent mask: D6, D8 dropped (indices 5, 7)")
masked_importance = sum(importances[i] for i in [5, 7])
kept_importance = sum(importances[i] for i in [0, 1, 2, 3, 4, 6])
print(f"Masked dims total importance: {masked_importance:.4f}")
print(f"Kept dims total importance:   {kept_importance:.4f}")

F8: n=51, dims=8
Best Y = 9.875265



RF feature importance (n=500 trees, max_depth=5):
  D1: 0.2270 ███████████ 
  D2: 0.0173  
  D3: 0.4324 █████████████████████ 
  D4: 0.0921 ████ 
  D5: 0.0468 ██ 
  D6: 0.0173  (MASKED)
  D7: 0.1417 ███████ 
  D8: 0.0255 █ (MASKED)

Current mask: D6, D8 dropped (indices 5, 7)
Masked dims total importance: 0.0428
Kept dims total importance:   0.9572


In [9]:
# --- Spearman correlations (all data and top-10 subset) ---
print("=== Spearman(dimension, Y) — Full data vs Top-10 ===\n")

top10_idx = np.argsort(Y8)[-10:]

print(f"{'Dim':<5} {'Full ρ':>8} {'Full p':>8} {'Top10 ρ':>8} {'Top10 p':>8}  {'Assessment':>20}")
print("-" * 65)

for i in range(dims8):
    rho_full, p_full = spearmanr(X8[:, i], Y8)
    rho_top, p_top = spearmanr(X8[top10_idx, i], Y8[top10_idx])

    if abs(rho_full) > 0.3 and p_full < 0.05:
        assessment = "IMPORTANT"
    elif abs(rho_full) < 0.1:
        assessment = "noise"
    else:
        assessment = "marginal"

    mask_flag = " ← masked" if i in [5, 7] else ""
    print(f"D{i+1:<4} {rho_full:>8.3f} {p_full:>8.4f} {rho_top:>8.3f} {p_top:>8.4f}  {assessment:>20}{mask_flag}")

=== Spearman(dimension, Y) — Full data vs Top-10 ===

Dim     Full ρ   Full p  Top10 ρ  Top10 p            Assessment
-----------------------------------------------------------------
D1      -0.644   0.0000   -0.588   0.0739             IMPORTANT
D2      -0.318   0.0231   -0.430   0.2145             IMPORTANT
D3      -0.698   0.0000   -0.830   0.0029             IMPORTANT
D4      -0.414   0.0026   -0.927   0.0001             IMPORTANT
D5       0.277   0.0487    0.697   0.0251              marginal
D6       0.008   0.9533   -0.042   0.9074                 noise ← masked
D7      -0.462   0.0007    0.103   0.7770             IMPORTANT
D8       0.230   0.1047    0.055   0.8810              marginal ← masked


In [10]:
# --- GP ARD length-scales: 8D vs 6D (masked) ---
print("=== GP ARD length-scales: 8D vs 6D (masked) ===\n")

# 8D GP
gp8_full = GaussianProcessRegressor(
    kernel=C(1.0) * Matern(length_scale=np.ones(8), nu=2.5),
    alpha=1e-6, normalize_y=False, n_restarts_optimizer=5,
)
gp8_full.fit(X8, Y8s)
ls_full = np.exp(gp8_full.kernel_.theta)

# 6D GP (masked)
dim_mask = [0, 1, 2, 3, 4, 6]
X8_masked = X8[:, dim_mask]
gp8_masked = GaussianProcessRegressor(
    kernel=C(1.0) * Matern(length_scale=np.ones(6), nu=2.5),
    alpha=1e-6, normalize_y=False, n_restarts_optimizer=5,
)
gp8_masked.fit(X8_masked, Y8s)
ls_masked = np.exp(gp8_masked.kernel_.theta)

print(f"{'Dim':<5} {'8D LS':>8} {'6D LS':>8}  Notes")
print("-" * 50)
j = 0
for i in range(8):
    ls8_val = ls_full[i + 1] if i + 1 < len(ls_full) else float('nan')
    if i in dim_mask:
        ls6_val = ls_masked[j + 1] if j + 1 < len(ls_masked) else float('nan')
        change = "shorter" if ls6_val < ls8_val * 0.8 else ("longer" if ls6_val > ls8_val * 1.2 else "similar")
        print(f"D{i+1:<4} {ls8_val:>8.3f} {ls6_val:>8.3f}  {change}")
        j += 1
    else:
        print(f"D{i+1:<4} {ls8_val:>8.3f}     ---  MASKED")

=== GP ARD length-scales: 8D vs 6D (masked) ===



Dim      8D LS    6D LS  Notes
--------------------------------------------------
D1       2.471    1.222  shorter
D2       4.891    0.783  shorter
D3       2.426    0.599  shorter
D4       3.289    1.102  shorter
D5      17.695    6.285  shorter
D6       2.821     ---  MASKED
D7       2.582    0.947  shorter
D8       3.676     ---  MASKED


---
## 5. Summary and W12 Recommendations

Consolidate findings across all four analyses into actionable recommendations
for the final 2 weeks.

In [11]:
print("=" * 70)
print("WEEK 12 SURROGATE ALTERNATIVES — SUMMARY")
print("=" * 70)

print("""
1. NN-64×2 for F6
   - NN mean LOO R² = 0.693 ± 0.019 vs GP = 0.690 → delta is +0.003
   - The preliminary scan's 0.734 was a single-seed artefact; across 5 seeds the
     NN and GP are statistically indistinguishable for F6.
   - NN suggestion dist=0.175 from best, GP dist=0.080, centroid dist=0.029.
   - VERDICT: NN does NOT improve over GP for F6. No surrogate switch warranted.

2. Local GP (TuRBO-style) for F3
   - LOCAL GP (k=15) LOO R² = 0.511 — dramatically better than global GP (-0.014)
   - Global GP gives pathological length-scales and argmax dist=0.606 from best.
   - Local GP (k=15) argmax dist=0.094 from best — 6.4× closer.
   - VERDICT: Local GP is the clear winner for F3. This is the TuRBO insight:
     fitting locally avoids pathological long length-scales.

3. Model-free weighted centroid
   - All centroids are very close to the best: F2=0.002, F4=0.008, F1=0.010.
   - F6 centroid dist=0.029 from best — closer than GP (0.131) or NN (0.175).
   - VERDICT: The centroid is a reliable sanity check and, for F6, the best
     candidate generator tested. Micro-perturbation of centroid is a safe strategy.

4. RF feature importance for F8
   - Masked dims D6 (0.017) and D8 (0.026) = 4.3% of total importance.
   - Spearman: D6 ρ=0.008 (p=0.95) = pure noise. D8 ρ=0.230 (p=0.10) = marginal.
   - 6D GP length-scales all < 6.3 vs 8D all > 2.5 — masking validated.
   - VERDICT: D6/D8 mask fully confirmed at n=51. No changes needed.
""")

print("Per-function W12 strategy:")
print("  F1: Model-free radial. Micro-perturbation near W8 hotspot [0.691, 0.707].")
print("  F2: Noise-dominated. Centroid [0.699, 0.935] is the safest bet.")
print("  F3: Local GP (k=15) or centroid. Much better than global GP.")
print("  F4: GP is best (R²=0.94). Centroid [0.438, 0.431, 0.362, 0.382].")
print("  F5: Continue gradient push D2/D3/D4 → 1.0. Mean/RBF working.")
print("  F6: GP or centroid [0.451, 0.416, 0.722, 0.776, 0.029]. NOT the NN.")
print("  F7: GP+RQ is best (R²=0.94). Search bounds + centroid as check.")
print("  F8: 6D GP confirmed. D6/D8 mask validated by RF importance.")

WEEK 12 SURROGATE ALTERNATIVES — SUMMARY

1. NN-64×2 for F6
   - NN mean LOO R² = 0.693 ± 0.019 vs GP = 0.690 → delta is +0.003
   - The preliminary scan's 0.734 was a single-seed artefact; across 5 seeds the
     NN and GP are statistically indistinguishable for F6.
   - NN suggestion dist=0.175 from best, GP dist=0.080, centroid dist=0.029.
   - VERDICT: NN does NOT improve over GP for F6. No surrogate switch warranted.

2. Local GP (TuRBO-style) for F3
   - LOCAL GP (k=15) LOO R² = 0.511 — dramatically better than global GP (-0.014)
   - Global GP gives pathological length-scales and argmax dist=0.606 from best.
   - Local GP (k=15) argmax dist=0.094 from best — 6.4× closer.
   - VERDICT: Local GP is the clear winner for F3. This is the TuRBO insight:
     fitting locally avoids pathological long length-scales.

3. Model-free weighted centroid
   - All centroids are very close to the best: F2=0.002, F4=0.008, F1=0.010.
   - F6 centroid dist=0.029 from best — closer than GP (0.131) o